# 2-Tier SAR Ship Classifier — Part 2
## Tier 2 Training + Evaluation + Summary

Requires Part 1 to have been run first (splits, label mappings, and `OUT_DIR` are carried over).

Tier 2 trains only on non-cargo ships. Each of the three architectures trains 4 models: os1, os2, fusar, joint.
At the end all weights and histories are zipped into a single timestamped archive.

### Note
Run all cells in **Part 1** first. The following variables must be in scope from that notebook before continuing here:
`OUT_DIR`, `WEIGHTS_DIR`, `MISC_DIR`, `DEVICE`, `tier2_domains`, `TIER2_CLASS_NAMES`, `TIER2_NUM_CLASSES`,
`get_model`, `train_model`, `evaluate_test`, `save_curves`, `free_cuda`, `FocalLoss`

## Tier 2 — ResNet-18

In [ ]:
resnet_t2_results = {}
resnet_t2_history = {}
resnet_dir = WEIGHTS_DIR / "resnet"
resnet_dir.mkdir(exist_ok=True)

print(f"{'='*55}")
print("  RESNET — Tier 2 (multi-class: non-cargo types)")
print(f"{'='*55}")

for domain_name, splits in tier2_domains.items():
    free_cuda()
    run   = f"resnet_tier2_{domain_name}"
    model = get_model("resnet", TIER2_NUM_CLASSES)
    model, history = train_model(model, run, TIER2_CLASS_NAMES,
                                 splits["train_loader"], splits["val_loader"], resnet_dir)
    resnet_t2_results[domain_name] = evaluate_test(model, splits["test_loader"], TIER2_CLASS_NAMES, run)
    resnet_t2_history[domain_name] = history
    free_cuda(model)

save_curves(resnet_t2_history, "ResNet — Tier 2", MISC_DIR / "resnet_tier2_curves.png")

## Tier 2 — ResNet-18 + CBAM (BAM)

In [ ]:
bam_t2_results = {}
bam_t2_history = {}
bam_dir = WEIGHTS_DIR / "bam"
bam_dir.mkdir(exist_ok=True)

print(f"{'='*55}")
print("  BAM — Tier 2 (multi-class: non-cargo types)")
print(f"{'='*55}")

for domain_name, splits in tier2_domains.items():
    free_cuda()
    run   = f"bam_tier2_{domain_name}"
    model = get_model("bam", TIER2_NUM_CLASSES)
    model, history = train_model(model, run, TIER2_CLASS_NAMES,
                                 splits["train_loader"], splits["val_loader"], bam_dir)
    bam_t2_results[domain_name] = evaluate_test(model, splits["test_loader"], TIER2_CLASS_NAMES, run)
    bam_t2_history[domain_name] = history
    free_cuda(model)

save_curves(bam_t2_history, "BAM — Tier 2", MISC_DIR / "bam_tier2_curves.png")

## Tier 2 — Swin Transformer

In [ ]:
swin_t2_results = {}
swin_t2_history = {}
swin_dir = WEIGHTS_DIR / "swin"
swin_dir.mkdir(exist_ok=True)

print(f"{'='*55}")
print("  SWIN — Tier 2 (multi-class: non-cargo types)")
print(f"{'='*55}")

for domain_name, splits in tier2_domains.items():
    free_cuda()
    run   = f"swin_tier2_{domain_name}"
    model = get_model("swin", TIER2_NUM_CLASSES)
    model, history = train_model(model, run, TIER2_CLASS_NAMES,
                                 splits["train_loader"], splits["val_loader"], swin_dir)
    swin_t2_results[domain_name] = evaluate_test(model, splits["test_loader"], TIER2_CLASS_NAMES, run)
    swin_t2_history[domain_name] = history
    free_cuda(model)

save_curves(swin_t2_history, "Swin — Tier 2", MISC_DIR / "swin_tier2_curves.png")

## Summary

In [ ]:
all_t2_results = {
    "resnet": resnet_t2_results,
    "bam"   : bam_t2_results,
    "swin"  : swin_t2_results,
}
all_t2_history = {
    "resnet": resnet_t2_history,
    "bam"   : bam_t2_history,
    "swin"  : swin_t2_history,
}

print("TIER 2 RESULTS")
print(f"{'Arch':<10}  {'Domain':<10}  {'Test Acc':>10}  {'Best Val':>10}")
print("-" * 48)
for arch, results in all_t2_results.items():
    for domain_name, res in results.items():
        best = max(all_t2_history[arch][domain_name]["val_acc"])
        print(f"{arch:<10}  {domain_name:<10}  {res['accuracy']:>9.2f}%  {best:>9.2f}%")

## Save History and Zip Outputs

In [ ]:
import json, zipfile

# Merge with Tier 1 history if it exists
tier1_path = MISC_DIR / "tier1_history.json"
tier1_history_loaded = {}
if tier1_path.exists():
    with open(tier1_path) as f:
        tier1_history_loaded = json.load(f)

combined_history = {
    arch: {
        "tier1": tier1_history_loaded.get(arch, {}),
        "tier2": all_t2_history[arch],
    }
    for arch in all_t2_history
}

with open(MISC_DIR / "all_history.json", "w") as f:
    json.dump(combined_history, f, indent=2)
print(f"History saved: {MISC_DIR / 'all_history.json'}")

# Zip everything
zip_path = OUT_DIR.parent / f"{OUT_DIR.name}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, file.relative_to(OUT_DIR.parent))

print(f"Archive: {zip_path}")